In [ ]:
import numpy as np
import pandas as pd
from glob import glob
from kneed import KneeLocator
from sklearn.isotonic import IsotonicRegression

In [ ]:
dataset = 'openLLMLeaderboard' # 'routerBench'

test = pd.read_pickle(f'../{dataset}/data/test.pkl')
train = pd.read_pickle(f'../{dataset}/data/train.pkl')

test.columns = (test.columns.str.lower().str.replace("__", "_"))
train.columns = (train.columns.str.lower().str.replace("__", "_"))

if dataset == 'routerBench':
    models = train.keys()[3:].tolist()
elif dataset == 'openLLMLeaderboard':
    models = train.keys()[2:-1].tolist()

models = [m.lower().replace("__", "_") for m in models]
models = sorted(models)
train = train[['prompt'] + models]
test = test[['prompt'] + models]

In [ ]:
#load data
RUNS = 10
raws=[]
banks_X=[]
banks_Y=[]
heatmaps=[]
for run in range(RUNS):
    preds = glob(f'../results/{dataset}/{run}/predictions/*')
    raw = pd.DataFrame()
    for pred in preds:
        name = pred.split('/')[-1].split('.npy')[0]
        name = name.lower().replace("__", "_")
        logits = np.load(pred)
        raw[name] = logits.squeeze().tolist()
    raw = raw[models]
    raws.append(raw)

    bank_preds = glob(f'../results/{dataset}/{run}/bank/*')
    bank = pd.DataFrame()
    for bank_pred in bank_preds:
        name = bank_pred.split('/')[-1].split('.npy')[0]
        name = name.lower().replace("__", "_")
        logits = np.load(bank_pred)
        bank[name] = logits.squeeze().tolist()
    bank = bank[models]
    banks_X.append(bank)

    temp = pd.read_pickle(f'../results/{dataset}/{run}/rejectors/bank.pkl')
    temp = temp.rename(columns=lambda c: c.lower().replace("__", "_"))
    temp = temp[models]
    banks_Y.append(temp)

    heatmaps.append(pd.read_pickle(f'../results/{dataset}/{run}/heatmap_compressed.pkl'))

In [ ]:
def compute_router_acc(selected_idx, arr):
    correct = arr[np.arange(len(arr)), selected_idx]
    acc = correct.mean()
    return acc

def calibration(raw, X, Y):    
    models = raw.columns.tolist()
    calibrated_raw = pd.DataFrame(index=raw.index, columns=models, dtype=float)
    
    for model in models:
        mask = ~Y[model].isna().values
        temp_Y = Y[model][mask]
        temp_X = X[model].iloc[mask]
        ir = IsotonicRegression(out_of_bounds='clip')
        try:
            ir.fit(temp_X, temp_Y)
            calibrated_raw[model] = ir.predict(raw[model].values)
        except:
            calibrated_raw[model] = raw[model].values
    return calibrated_raw

def raw_router(raw):
    best_idx = np.argmax(raw, axis=1)
    return best_idx

def elbow_router(raw, to_prune=raw, test = False):
    best_idx = raw_router(raw)
    unique, counts = np.unique(best_idx, return_counts=True)
    order = np.argsort(counts)[::-1] 
    unique_sorted = unique[order]      
    counts_sorted = counts[order] 
    x = np.arange(len(counts_sorted))
    
    knee = KneeLocator(x, counts_sorted, curve='convex', direction='decreasing').knee
    if knee is None or knee == 0:
        knee = len(best_idx)
    x_clip = unique_sorted[:knee]

    if test:
        print(len(x_clip))
        print(x_clip)
    
    pruned_raw = to_prune.iloc[:, x_clip]
    idx_local = np.argmax(pruned_raw, axis=1)
    idx_global = x_clip[idx_local]
    return idx_global

def best_models(test, models):
    accuracies = (test[models].mean().sort_values(ascending=False))
    return accuracies

def reduce_redundancy(M, tau, K):
    models = models = sorted(M.columns)
    A = M.loc[models, models].copy()
    C = (A.values >= tau)

    uncovered = set(range(len(models)))
    cov = []
    selected = []

    for k in range(1, K+1):
        if uncovered:
            uncovered_mask = np.array([j in uncovered for j in range(len(models))])
            gains = C[:, uncovered_mask].sum(axis=1)
            best_i = int(np.argmax(gains))
            if gains[best_i] > 0:
                selected.append(models[best_i])
                newly_covered = np.where(C[best_i])[0]
                uncovered -= set(newly_covered.tolist())
                C[best_i, :] = False

        covered_frac = 1.0 - (len(uncovered) / len(models))
        cov.append(covered_frac)

    return np.array(cov), selected

def prune_redundancy_router(raw, heatmap_df, tau, K, test=True):
    M = heatmap_df.pivot(index='source', columns='target', values='auroc')
    M = M.fillna(M.median(axis=1), axis=0)

    _, selected_models = reduce_redundancy(M, tau, K)
    selected_models = [model.lower().replace('__', '_') for model in selected_models]

    if test:
        print(len(selected_models))
        print(selected_models)

    raw_selected = raw[selected_models]
    idx_local = np.argmax(raw_selected, axis=1)
    chosen_models = raw_selected.columns[idx_local]
    global_idx = [raw.columns.get_loc(m) for m in chosen_models]
    global_idx = np.array(global_idx, dtype=int)
    return global_idx

def compute_best_tau(X, Y, heatmap, K=20):
    arr = Y.to_numpy()
    taus = np.arange(0.01, 1.01, 0.01)
    temp=[]
    for tau in taus:
        try:
            pruned_idx = prune_redundancy_router(X, heatmap, tau, K, False)
            temp.append(compute_router_acc(pruned_idx, arr))
        except:
            temp.append(0)
    idx = temp.index(max(temp))
    best_tau = taus[idx]
    return best_tau

In [ ]:
arr = test[models].to_numpy()
res = []

for run in range(RUNS):
    temp = []
    raw_router_idx = raw_router(raws[run])
    temp.append(compute_router_acc(raw_router_idx, arr))
    
    cal = calibration(raws[run], banks_X[run], banks_Y[run])
    cal_router_idx = raw_router(cal)
    temp.append(compute_router_acc(cal_router_idx, arr))
    
    elbow_router_idx = elbow_router(raws[run])
    temp.append(compute_router_acc(elbow_router_idx, arr))

    best_tau = compute_best_tau(banks_X[run], banks_Y[run], heatmaps[run])
    pruned_idx = prune_redundancy_router(raws[run], heatmaps[run], best_tau, len(models), False)
    temp.append(compute_router_acc(pruned_idx, arr))

    res.append(temp)

In [ ]:
#load baseline (KNN / MLP) results
baselines = glob(f'../results/baselines/{dataset}/*.npz')

predicted_models_mlp = []
predicted_models_knn = []
mlp_acc = []
knn_acc = []
mlp_layers = []
for baseline in baselines:
    data = np.load(baseline, allow_pickle=True)
    mlp_layers.append(baseline.split('_')[-1].split('.')[0])
    
    predicted_models_mlp.append(data["predicted_models_mlp"])
    predicted_models_knn.append(data["predicted_models_knn"])
    mlp_acc.append(data["mlp_acc"].item())
    knn_acc.append(data["knn_acc"].item())

baseline_scores = pd.DataFrame({
    "mlp_acc": mlp_acc,
    "knn_acc": knn_acc,
    "mlp_layers": mlp_layers
})

In [ ]:
res_df = pd.DataFrame(res, columns=['raw', 'cal', 'elbow-pruning', 'coverage-pruning'])
res_df['best model'] = best_models(test, models).iloc[0]
res_df['oracle'] = test[models].max(axis=1).mean()

mlp_mean_by_layer = baseline_scores.groupby("mlp_layers")["mlp_acc"].mean()
for layer, mean_acc in mlp_mean_by_layer.items():
    res_df[f"mlp_{layer}"] = mean_acc
res_df['knn'] = baseline_scores['knn_acc'].mean()
res_df.mean().sort_values()

In [ ]:
#Pruning Counts
#(24+24+22+23+25+22+24+14+24+15)/10 #OpenLLMLeaderboard
#(4+4+4+4+4+4+3+4+4+4)/10 #RouterBench

#(9+10+16+11+8+11+13+5+21+21)/10 #OpenLLMLeaderboard
#(2+2+2+1+2+2+2+2+2+2)/10 #RouterBench